# Building the entire data preprocessing pipeline for LLM training from scratch

It consists of 4 main steps:
1. Tokenization (Word based, Sub-word based, Character based)
2. Token Embeddings (Converting the token Ids into vector embeddings)
3. Positional Embeddings (Encoding information about the position of the words in the sentance)
4. Input Embeddings = Token Embeddings + Positional Embeddings (Final embeddings which also contain the information about the position of the word in the sentence which is ready to fed to the LLM for training)

In [101]:
import tiktoken

In [102]:
tokenizer=tiktoken.get_encoding("gpt2")
with open('harrypotter.txt', 'r') as harryPotterBook:
    bookContent=harryPotterBook.read()
vocabulary=tokenizer.encode(bookContent, allowed_special={"<|endoftext|>"})
print(bookContent[:100])

Harry Potter and the Sorcerer's Stone 

CHAPTER ONE 

THE BOY WHO LIVED 

Mr. and Mrs. Dursley, of n


## Implementing DataLoader and Dataset classes from PyTorch for efficient loading of data

In [103]:
import torch
from torch.utils.data import DataLoader, Dataset

In [104]:
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        self.input_ids=[]
        self.target_ids=[]

        # Tokenization: Converting the raw text into individual tokens using BPE algorithm 
        encoded_text=tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        # Creating the input and target pairs
        for i in range(0, len(encoded_text)-context_length, stride):
            input_id=encoded_text[i:i+context_length]
            target_id=encoded_text[i+1:i+context_length+1]
            self.input_ids.append(torch.tensor(input_id))
            self.target_ids.append(torch.tensor(target_id))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [105]:
def createDataloaderV1(text, batch_size=4, context_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer=tiktoken.get_encoding("gpt2")

    # Create the dataset
    dataset=GPTDatasetV1(text, tokenizer, context_length, stride)

    # Create the Dataloader
    dataloader=DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

## Building the input and target tensors using the dataloader with batch_size=8, context_length=256 and stride=128

In [106]:
context_length=256
dataloader=createDataloaderV1(
    text=bookContent,
    batch_size=8,
    context_length=context_length,
    stride=128,
    shuffle=False
)

In [107]:
data_iterator=iter(dataloader)

In [108]:
first_batch=next(data_iterator)
first_batch

[tensor([[18308, 14179,   290,  ...,   484,  8020,   470],
         [   13,  9074,    13,  ...,  9074,    13,   360],
         [ 1138,   329,  1811,  ...,   319,   262, 19353],
         ...,
         [16809,  9974,    11,  ..., 48508,  6568,   306],
         [  286, 26721,   339,  ...,   736,   284,   262],
         [ 1978,    13,  1770,  ...,   618,   339,  1807]]),
 tensor([[14179,   290,   262,  ...,  8020,   470,  1138],
         [ 9074,    13,   360,  ...,    13,   360,  1834],
         [  329,  1811,   812,  ...,   262, 19353,    11],
         ...,
         [ 9974,    11,   475,  ...,  6568,   306,  1978],
         [26721,   339,   373,  ...,   284,   262,  4324],
         [   13,  1770,    13,  ...,   339,  1807,   339]])]

## Token Embeddings

In [109]:
vocab_size=len(vocabulary)
output_dims=12288 # The GPT-3 model used 12,288 dimensions
print(f"Vocabulary size: {vocab_size}\nOutput Dimensions: {output_dims}")

Vocabulary size: 113828
Output Dimensions: 12288


In [110]:
torch.manual_seed(123)
token_embedding_layer=torch.nn.Embedding(vocab_size, output_dims)

In [111]:
print(token_embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035,  ...,  1.2624, -0.3365,  0.4225],
        [ 1.7306,  0.1500,  0.2027,  ...,  0.4064, -0.4669,  0.4912],
        [ 0.6639, -2.1862,  0.0922,  ...,  0.1715, -0.0195, -0.7284],
        ...,
        [ 0.2089,  0.6577,  0.8833,  ..., -2.1363,  0.7559,  1.0994],
        [ 0.5712, -0.5044,  1.6612,  ..., -1.6438, -0.4522,  1.7123],
        [ 0.8865, -0.4949, -0.7625,  ...,  0.1436,  2.6445, -0.4684]],
       requires_grad=True)


## Input ids to Embedd

In [112]:
inputs, targets = next(dataIterator)
print(f"Input IDs:\n{inputs}")
print(f"\nInput Shape: {inputs.shape}")

Input IDs:
tensor([[ 5850,  4385,   326,  ...,   339, 16555,   339],
        [   11,   262,  1862,  ...,   632,  2492,   470],
        [ 8020,   470,   531,  ...,   683,  1194,   530],
        ...,
        [18577,   340,   656,  ...,   351,  5850,   338],
        [  992,  1497,    13,  ...,   198,     1,  8496],
        [   13,   220,   198,  ...,  8322, 25173,     0]])

Input Shape: torch.Size([8, 256])


## Embeddings of these input ids

In [113]:
token_embeddings=token_embedding_layer(inputs)
print(f"Token Embeddings:\n{token_embeddings}")
print(f"\nEmbeddings shape: {token_embeddings.shape}")

Token Embeddings:
tensor([[[-1.0008,  0.9301,  3.1501,  ..., -1.2997,  1.4784, -0.5752],
         [ 0.4912,  0.0954,  0.0525,  ..., -0.0386, -0.9316, -0.8276],
         [-1.2710, -0.4447,  2.3075,  ..., -0.0826, -0.2010,  0.0985],
         ...,
         [ 0.6896, -0.4072,  1.7801,  ..., -0.4584, -0.8939,  0.5864],
         [-0.5670, -0.3591, -1.0783,  ..., -0.8952, -0.3076, -0.1534],
         [ 0.6896, -0.4072,  1.7801,  ..., -0.4584, -0.8939,  0.5864]],

        [[ 1.1162, -0.9649, -0.1393,  ..., -0.5116, -0.1316, -0.0337],
         [ 0.9405,  0.8932, -1.1660,  ...,  0.0585, -0.5071,  2.4862],
         [ 1.1929, -0.3857,  0.1325,  ...,  0.1141, -1.2133,  0.7556],
         ...,
         [ 0.1966, -2.9774,  0.1625,  ...,  0.8738, -1.6631,  0.5715],
         [-0.7013, -0.5026,  0.0761,  ..., -0.5636,  0.5707,  1.3207],
         [-0.2740,  1.3380, -0.9158,  ...,  1.4837,  1.1244,  0.2956]],

        [[ 0.2593,  0.3065, -0.2861,  ...,  0.3202,  1.4044, -1.1095],
         [-0.2740,  1.3380,

### Here we have got a tensor of 8X256X12288 dimensions where 8 is the batch_size, 256 is the context_length and 12288 is the dimensions

## Positional Embeddings

In [114]:
positional_embedding_layer=torch.nn.Embedding(context_length, output_dims)

In [115]:
positional_embeddings=positional_embedding_layer(torch.arange(context_length))
print(f"Positional Embeddings:\n{positional_embeddings}")
print(f"\nEmbeddings Shape: {positional_embeddings.shape}")

Positional Embeddings:
tensor([[-0.3203,  0.0965, -0.6700,  ..., -0.0275, -1.2926, -1.3775],
        [-1.6376,  0.8396, -1.1540,  ..., -0.1359,  2.3471,  0.6358],
        [ 0.2029,  0.7857, -0.2219,  ..., -0.5848,  0.5402, -1.7874],
        ...,
        [-1.2802, -0.0967,  1.3683,  ...,  1.3389,  1.1215,  0.5198],
        [-1.4399, -0.2652,  1.2920,  ...,  1.8741, -0.2989,  0.7707],
        [-0.0195,  1.2655,  0.0833,  ...,  0.6789, -0.6330, -1.6648]],
       grad_fn=<EmbeddingBackward0>)

Embeddings Shape: torch.Size([256, 12288])


## Input Embeddings = Token Embeddings + Positional Embeddings

In [116]:
input_embeddings=token_embeddings+positional_embeddings
print(f"Input Embeddings:\n{input_embeddings}")
print(f"\nEmbeddings shape: {input_embeddings.shape}")

Input Embeddings:
tensor([[[-1.3211,  1.0265,  2.4801,  ..., -1.3272,  0.1858, -1.9527],
         [-1.1464,  0.9350, -1.1015,  ..., -0.1744,  1.4155, -0.1917],
         [-1.0681,  0.3410,  2.0856,  ..., -0.6673,  0.3392, -1.6890],
         ...,
         [-0.5906, -0.5039,  3.1484,  ...,  0.8805,  0.2276,  1.1062],
         [-2.0069, -0.6244,  0.2137,  ...,  0.9789, -0.6065,  0.6172],
         [ 0.6702,  0.8584,  1.8634,  ...,  0.2204, -1.5269, -1.0784]],

        [[ 0.7959, -0.8684, -0.8093,  ..., -0.5391, -1.4242, -1.4112],
         [-0.6971,  1.7329, -2.3200,  ..., -0.0773,  1.8400,  3.1221],
         [ 1.3958,  0.3999, -0.0894,  ..., -0.4707, -0.6731, -1.0319],
         ...,
         [-1.0836, -3.0741,  1.5308,  ...,  2.2127, -0.5416,  1.0913],
         [-2.1413, -0.7678,  1.3682,  ...,  1.3105,  0.2717,  2.0913],
         [-0.2935,  2.6035, -0.8325,  ...,  2.1626,  0.4914, -1.3692]],

        [[-0.0610,  0.4030, -0.9561,  ...,  0.2927,  0.1118, -2.4870],
         [-1.9117,  2.1776,

# Now these Input Embeddings can be fed to Large Language Model for training